1. Reiniciar completamente Colab

Seleccione:

Entorno de ejecución → Desconectar y eliminar entorno de ejecución

Luego:

Entorno de ejecución → Cambiar tipo de entorno de ejecución

Seleccione:

Acelerador: GPU
Versión del entorno: 2025.07

In [ ]:
2. Verificar Python

In [ ]:
# ============================================================
# VERIFICAR PYTHON
# ============================================================

import sys

print("Python:", sys.version)

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "Debe utilizar el entorno 2025.07 de Colab "
        "con Python 3.11."
    )

print("✅ Python 3.11 detectado.")

Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
✅ Python 3.11 detectado.


In [ ]:
# ============================================================
# INSTALAR EFFICIENTDET MODEL MAKER
# SOLO COMPONENTES DE DETECCIÓN
# ============================================================

import subprocess
import sys


def ejecutar_pip(*argumentos):
    """
    Ejecuta pip y detiene inmediatamente la celda
    cuando ocurre cualquier error.
    """

    comando = [
        sys.executable,
        "-m",
        "pip",
        *argumentos
    ]

    print("\nEjecutando:")
    print(" ".join(comando))

    subprocess.run(
        comando,
        check=True
    )


# Actualizar herramientas básicas
ejecutar_pip(
    "install",
    "--upgrade",
    "pip==23.3.2",
    "setuptools==68.2.2",
    "wheel",
    "jedi"
)


# TensorFlow y dependencias compatibles con Python 3.11
ejecutar_pip(
    "install",
    "--no-cache-dir",
    "--upgrade",

    "numpy==1.23.5",
    "protobuf==3.20.3",

    "tensorflow==2.15.1",
    "keras==2.15.0",

    "tflite-support==0.4.4",

    "tf-models-official==2.15.0",
    "tensorflow-addons==0.23.0",
    "tensorflow-model-optimization==0.8.0",
    "tensorflow-hub==0.15.0",
    "tensorflow-datasets==4.9.3",

    "neural-structured-learning==1.4.0",

    "pycocotools==2.0.7",
    "Cython==3.0.11",

    "pandas==2.0.3",
    "scikit-learn==1.3.2",
    "matplotlib==3.7.5",

    "Pillow==10.4.0",
    "PyYAML==6.0.2",
    "lxml==4.9.4",
    "fire==0.5.0",
    "flatbuffers==23.5.26"
)


# Instalar Model Maker sin resolver sus dependencias antiguas
ejecutar_pip(
    "install",
    "--no-cache-dir",
    "--no-deps",
    "tflite-model-maker==0.4.3"
)

print("\n✅ Paquetes instalados.")


Ejecutando:
/usr/bin/python3 -m pip install --upgrade pip==23.3.2 setuptools==68.2.2 wheel jedi

Ejecutando:
/usr/bin/python3 -m pip install --no-cache-dir --upgrade numpy==1.23.5 protobuf==3.20.3 tensorflow==2.15.1 keras==2.15.0 tflite-support==0.4.4 tf-models-official==2.15.0 tensorflow-addons==0.23.0 tensorflow-model-optimization==0.8.0 tensorflow-hub==0.15.0 tensorflow-datasets==4.9.3 neural-structured-learning==1.4.0 pycocotools==2.0.7 Cython==3.0.11 pandas==2.0.3 scikit-learn==1.3.2 matplotlib==3.7.5 Pillow==10.4.0 PyYAML==6.0.2 lxml==4.9.4 fire==0.5.0 flatbuffers==23.5.26

Ejecutando:
/usr/bin/python3 -m pip install --no-cache-dir --no-deps tflite-model-maker==0.4.3

✅ Paquetes instalados.


3. Instalar solo las dependencias necesarias para detección

In [ ]:
# ============================================================
# AJUSTAR TFLITE MODEL MAKER PARA OBJECT DETECTION
# ============================================================

import site
from pathlib import Path


carpeta_paquete = None

rutas_posibles = (
    site.getsitepackages()
    + [site.getusersitepackages()]
)

for ruta_base in rutas_posibles:

    candidato = (
        Path(ruta_base)
        / "tflite_model_maker"
    )

    if candidato.exists():
        carpeta_paquete = candidato
        break


if carpeta_paquete is None:
    raise RuntimeError(
        "No se encontró la instalación de "
        "tflite_model_maker."
    )


archivo_init = (
    carpeta_paquete
    / "__init__.py"
)


# Guardar una copia del archivo original
archivo_respaldo = (
    carpeta_paquete
    / "__init___original.py"
)

if archivo_init.exists():

    archivo_respaldo.write_text(
        archivo_init.read_text(
            encoding="utf-8"
        ),
        encoding="utf-8"
    )


# Crear un inicio mínimo
contenido_init = '''
"""
Inicialización mínima de TFLite Model Maker.

Configurada exclusivamente para entrenamiento
de detectores de objetos EfficientDet-Lite.
"""

__version__ = "0.4.3"
'''.strip()


archivo_init.write_text(
    contenido_init + "\n",
    encoding="utf-8"
)


print(
    "✅ Inicialización de Model Maker modificada."
)

print(
    "Ubicación:",
    carpeta_paquete
)

print(
    "\nAhora reinicie la sesión de Colab."
)

✅ Inicialización de Model Maker modificada.
Ubicación: /usr/local/lib/python3.11/dist-packages/tflite_model_maker

Ahora reinicie la sesión de Colab.


5. Reiniciar solamente la sesión

Seleccione:

Entorno de ejecución → Reiniciar sesión

No elija nuevamente “eliminar entorno”, porque borraría la instalación.

6. Nueva celda de verificación

In [ ]:
# ============================================================
# VERIFICAR MODEL MAKER Y EFFICIENTDET-LITE1
# ============================================================

import sys
import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)


from tflite_model_maker import object_detector

from tflite_model_maker.config import (
    QuantizationConfig
)


# Crear la especificación directamente,
# sin importar model_spec
spec_prueba = (
    object_detector.EfficientDetLite1Spec()
)


print(
    "✅ object_detector importado correctamente."
)

print(
    "✅ EfficientDet-Lite1 disponible."
)

print(
    "✅ QuantizationConfig disponible."
)


gpus = tf.config.list_physical_devices(
    "GPU"
)

if gpus:
    print("✅ GPU detectada:", gpus)
else:
    print(
        "⚠ Model Maker funciona, "
        "pero TensorFlow no detectó GPU."
    )

AttributeError: module 'numpy' has no attribute 'dtypes'